# 在 Colab GPU 上运行 AlexNet

这份 notebook 在 VS Code 中连接 Colab GPU 内核使用。它会同步 GitHub 项目并运行项目中已有的 Python 脚本。

## 使用前准备
1. 在 VS Code 中连接 Colab kernel，并确认运行时选择了 GPU。
2. 这是公开仓库，不需要 GitHub Token 或 Secret。
3. 下面的代码必须按顺序执行，每一步先确认出现 `[OK]`，再执行下一步。

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 设置 GitHub 仓库地址，以及 Colab 云端的临时工作目录。
PROJECT_URL = "https://github.com/espstarry/pytorch-learning.git"
PROJECT_DIR = Path("/content/pytorch-learning")

# 如果上次克隆失败留下了残缺目录，先删除它。
if PROJECT_DIR.exists() and not (PROJECT_DIR / ".git").exists():
    shutil.rmtree(PROJECT_DIR)

# 第一次运行时克隆项目；之后运行时拉取 GitHub 最新代码。
if not PROJECT_DIR.exists():
    if not PROJECT_URL:
        raise ValueError("Set PROJECT_URL to your GitHub repository URL, then run this cell again.")
    clone_url = PROJECT_URL
    result = subprocess.run(["git", "clone", clone_url, str(PROJECT_DIR)], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"项目克隆失败：\n{error}\n请检查仓库地址和 Token 权限，然后重新运行本单元格。")
else:
    clone_url = PROJECT_URL
    subprocess.run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", clone_url], check=True)
    result = subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"项目更新失败：\n{error}\n请检查仓库地址和 Token 权限，然后重新运行本单元格。")
    print(result.stdout.strip() or "项目已经是最新版本")

os.chdir(PROJECT_DIR)
print("项目目录：", Path.cwd())
print("[OK] 项目准备完成")

In [ ]:
# 检查 PyTorch、CUDA 和 Colab 分配的 GPU。
import torch

print("PyTorch 版本：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))
else:
    print("没有连接 GPU，请在 Colab 中将运行时切换为 GPU。")

!nvidia-smi
print("[OK] GPU 检查完成")

## 1. Smoke test：模型通路检查

这一步使用一张随机图片，只执行一次前向传播、反向传播和参数更新，不会训练整个数据集。
预期输出包含 `device: cuda`、`input: [1, 3, 224, 224]`、`output: [1, 10]` 和 `[OK] Smoke test passed`。

In [ ]:
# 进入 AlexNet 示例目录，执行一次最小前向/反向测试。
os.chdir(PROJECT_DIR / "demos/vision/alexnet_110")
!python train.py && echo "[OK] Smoke test passed"

## 2. CIFAR-10 训练

训练使用 Hugging Face 上的 `uoft-cs/cifar10` 数据集，数据会缓存到当前 Colab 运行时。

当前只使用 1000 张训练图片、200 张测试图片和 1 个 epoch，用来先验证流程。训练结束会生成 `loss_data.json`。确认正常后，再进行下面的正式实验。

In [ ]:
# 安装 Hugging Face 数据集读取库。
!pip install -q datasets
import datasets
print("datasets 版本：", datasets.__version__)
print("[OK] Hugging Face 数据集库准备完成")


运行下面的代码开始训练。训练过程中每个 epoch 会输出一次测试准确率。
预期输出类似 `epoch 1: train_loss=... test_loss=... test_accuracy=...`，最后出现 `[OK] CIFAR-10 训练完成`。
训练结束后，打开本地 `loss_viewer/viewer.html`，拖入下载的 `loss_data.json` 查看曲线。

In [ ]:
# 先运行一个很小的学习实验，确认数据、模型和 GPU 都能连通。
!python train_cifar10.py --source huggingface --epochs 1 --train-samples 1000 --test-samples 200 && echo "[OK] CIFAR-10 训练完成"
import base64
from IPython.display import HTML, display
display(HTML('<a download="loss_data.json" href="data:application/json;base64,' + base64.b64encode(Path("loss_data.json").read_bytes()).decode() + '">点击下载 loss_data.json</a>'))

## 3. CIFAR-10 分类标签
CIFAR-10 一共有 10 个类别。训练输出中的标签编号对应下面的类别：
`0 airplane`、`1 automobile`、`2 bird`、`3 cat`、`4 deer`、`5 dog`、`6 frog`、`7 horse`、`8 ship`、`9 truck`。

In [ ]:
# 打印 CIFAR-10 的数字标签与类别名称对应关系。
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
for label, name in enumerate(class_names):
    print(f"{label}: {name}")
print("[OK] 分类标签已加载")

## 4. 正式学习实验
前面的 1 epoch 只是流程验证，只有一个曲线点。下面运行 10 个 epoch，并使用更多样本，才能观察 train loss、test loss 和 test accuracy 的变化。
这一步可能需要几分钟到几十分钟，取决于 Colab GPU 和 Hugging Face 下载速度。运行过程中可以观察每个 epoch 的输出，也可以点击停止按钮终止。
预期现象：train loss 通常逐步下降，test accuracy 通常逐步上升；如果两者背离，可能出现过拟合。

In [ ]:
# 正式实验：10 个 epoch，使用 10,000 张训练图片和 2,000 张测试图片。
!python train_cifar10.py --source huggingface --epochs 10 --train-samples 10000 --test-samples 2000
import base64
from IPython.display import HTML, display
display(HTML('<a download="loss_data.json" href="data:application/json;base64,' + base64.b64encode(Path("loss_data.json").read_bytes()).decode() + '">点击下载正式实验 loss_data.json</a>'))

## 5. 导出 Tensor 并在本地可视化

这一步运行 AlexNet 的前向传播，并将中间 Tensor 写入 `tensor_data.json`。代码完成后会自动下载 JSON 文件。
然后打开本地 Tensor Viewer，把这个 JSON 文件拖入网页。预期输出是 Tensor 导出完成和文件下载提示。

In [ ]:
# 导出 AlexNet 各层的中间 Tensor，并提供本地下载链接。
!python export_tensors.py && echo "[OK] Tensor 导出完成"
import base64
from IPython.display import HTML, display
data = base64.b64encode(Path("tensor_data.json").read_bytes()).decode()
link = f'<a download="tensor_data.json" href="data:application/json;base64,{data}">点击下载 tensor_data.json</a>'
display(HTML(link))
print("[OK] 请点击上面的下载链接保存到本地")